# 1、content的使用
消息的 content 可以理解为数据内容，它是弱类型的，支持字符串和列表（列表元素通常为字典）。
举例1：
保存多模态的方式

In [24]:
import base64
from langchain_core.messages import HumanMessage
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from pydantic import SecretStr
# 初始化
# 1、读取.env配置文件中的信息。相关的环境变量以.env文件优先
load_dotenv(verbose=True)
OPENROUTER_API_KEY=SecretStr(os.getenv("OPENROUTER_API_KEY")or"")

# 2、模型的初始化
model=ChatOpenRouter(
    model="nvidia/nemotron-3.5-super-vl:free",
    api_key=OPENROUTER_API_KEY,

)
def encode_image(img_path, img_type='jpg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"
# 图像路径
img_path = "image_test.jpg"
# 获取图像base64编码字符串
base64_image = encode_image(img_path)
response = model.invoke(
[
HumanMessage(
content=[
{'type': 'text', 'text': '这张图里有什么？'},
{
'type': 'image_url',
"image_url": {"url": base64_image},
}
    ]
        )
        ]
            )
print(response.content)

这张图是一幅卡通风格的插图，场景充满了超现实和奇思妙想的色彩。

画面中包含以下主要元素：

1.  **一位特殊的画家：** 图片中央有一个背对着观众的人，正坐在画架旁作画。他穿着紫色的外套和绿色的靴子。
2.  **向日葵替代头部：** 最引人注目的是，这位画家脖子上并没有头顶，而是直接连接着一株高大的向日葵，向日葵花盘朝上，仿佛就是他的头。
3.  **绘画动作：** 画家正拿着画笔在画布上作画，画布上画着几个排列成弧线的小圆圈，可能在勾勒天空中的太阳。
4.  **背景景色：**
    *   **天空：** 上方有一个明亮的圆形太阳。
    *   **远景：** 中间是一片平静的水面，右侧远处有一座黑色的山峦剪影。
    *   **地面：** 画家坐在沙地上，地上有一些简单的草丛。

整体来看，这幅画构思独特，将自然（向日葵、太阳）与艺术创作融为一体，传达出一种关于艺术、生长和对光明追求的意象。


## 2、content block的使用
在 LangChain 1.x 中， content_blocks 是消息对象（BaseMessage）的一项重大升级。它的核心目标
是提供一种跨模型供应商、标准化的多模态数据结构。

过去，处理图片、音频、甚至是模型生成的“思维链（Reasoning）”内容时，不同供应商（OpenAI,Anthropic, Google 等）的 API 格式各异，导致开发者需要写大量的适配代码。 content_blocks 的出
现终结了这种混乱。
在 LangChain 1.2 版本中，消息对象的 content 属性依然存在（为了向前兼容），但新增了
content_blocks 属性，可以将 content 解析为标准、类型安全的表示。
- 数据结构：它是一个 list[TypedDict] 。
- 统一格式：每个 block 都有一个 type 字段，用于区分内容类型。
- 支持类型：包括 text （文本）、 image （图片）、 audio （音频）、 video （视频）、tool_call （工具调用）以及 reasoning （推理/思维链）。

借助 content_blocks ，我们可以用一套标准代码，无缝地在不同厂商的模型之间切换。



举例1 输入格式化

In [27]:
import base64
from langchain_core.messages import HumanMessage
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from pydantic import SecretStr
# 初始化
# 1、读取.env配置文件中的信息。相关的环境变量以.env文件优先
load_dotenv(verbose=True)
OPENROUTER_API_KEY=SecretStr(os.getenv("OPENROUTER_API_KEY")or"")

# 2、模型的初始化
model=ChatOpenRouter(
    model="nvidia/nemotron-3.5-super-vl:free",
    api_key=OPENROUTER_API_KEY,

)
def encode_image(img_path, img_type='jpg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        # 在使用 LangChain 的 content_blocks 标准格式（指定 'type': 'image'）时，'base64' 字段只接受纯 Base64 编码字符串
        return base64.b64encode(img_file.read()).decode("utf-8")
# 图像路径
img_path = "image1.png"
# 获取图像base64编码字符串
base64_image = encode_image(img_path)
response = model.invoke(
[
HumanMessage(
# 推荐的统一写法
    content_blocks=[
        {'type': 'text', 'text': '这张图里有什么？'},
{
        'type': 'image',
        'base64': base64_image,
        'mime_type': 'image/png',
}
    ]
            )]
)
print(response.content)

这张图片主要展示了一只橘色猫咪的特写面部。猫的鼻子和嘴巴周围有白色的毛发，正直视镜头。在图片的左下角，有一只模糊的人类手指伸入画面。背景比较模糊，看起来像是室内环境。


举例2 输出格式化


content_blocks 还可用于输出格式化，其输出包含思考
内容，后者位于 additional_kwargs 的 reasoning_content 字段下。比如：

In [28]:
import base64
from langchain_core.messages import HumanMessage
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from pydantic import SecretStr
from rich import print as rprint
# 初始化
# 1、读取.env配置文件中的信息。相关的环境变量以.env文件优先
load_dotenv(verbose=True)
OPENROUTER_API_KEY=SecretStr(os.getenv("OPENROUTER_API_KEY")or"")

# 2、模型的初始化
model=ChatOpenRouter(
    model="nvidia/nemotron-3.5-super-vl:free",
    api_key=OPENROUTER_API_KEY,

)
#向大模型发送单条数据
response=model.invoke("用一句话介绍你自己")
rprint('=' * 20, '-> response <-', '=' * 20)
rprint(response)
rprint('=' * 20, '-> response.content <-', '=' * 20)
rprint(response.content)
rprint('=' * 20, '-> response.content_blocks <-', '=' * 20)
rprint(response.content_blocks)

==================== -> response <- ====================

AIMessage(
    content='我是一个人工智能语言助手，可以帮你回答问题、提供信息，并协助完成写作、翻译、编程等各类文字任务。',
    additional_kwargs={
        'reasoning_content': 
'用户要求我用一句话介绍自己。我需要简洁、准确地说明自己是一个AI助手，由NVIDIA的研究人员训练，但这里不需要提具体公司
背景，除非我知道。实际上我是由NVIDIA的研究人员训练的语言模型。不过用户只要求一句话。\n\n可以这样说：我是一个由人工
智能驱动的对话助手，能够帮助你回答问题、提供信息并完成各种文字任务。\n\n或者更简单：我是一个人工智能语言模型，可以
协助你完成写作、翻译、编程、问答等各类任务。\n\n用中文，一句话。',
        'reasoning_details': [
            {
                'type': 'reasoning.text',
                'text': 
'用户要求我用一句话介绍自己。我需要简洁、准确地说明自己是一个AI助手，由NVIDIA的研究人员训练，但这里不需要提具体公司
背景，除非我知道。实际上我是由NVIDIA的研究人员训练的语言模型。不过用户只要求一句话。\n\n可以这样说：我是一个由人工
智能驱动的对话助手，能够帮助你回答问题、提供信息并完成各种文字任务。\n\n或者更简单：我是一个人工智能语言模型，可以
协助你完成写作、翻译、编程、问答等各类任务。\n\n用中文，一句话。',
                'format': 'unknown',
                'index': 0.0
            }
        ]
    },
    response_metadata={
        'model_name': 'nvidia/nemotron-3.5-super-vl:free',
        'id': 'gen-1793153399-Z6jDdnu0ufioYDpSHDL3',
        'created': 1793153399,
        'object': 'chat.completion',
        'finish_reason': 'stop',
        'logprobs': None,
        'model_provider': 'openrouter'
    },
    id='lc_run--01a18045-78ea-7fc3-9864-b74d9bbfbba1-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 23,
        'output_tokens': 210,
        'total_tokens': 233,
        'input_token_details': {'cache_read': 0, 'cache_creation': 0},
        'output_token_details': {'reasoning': 167}
    }
)

==================== -> response.content <- ====================

我是一个人工智能语言助手，可以帮你回答问题、提供信息，并协助完成写作、翻译、编程等各类文字任务。

==================== -> response.content_blocks <- ====================

[
    {
        'type': 'reasoning',
        'reasoning': 
'用户要求我用一句话介绍自己。我需要简洁、准确地说明自己是一个AI助手，由NVIDIA的研究人员训练，但这里不需要提具体公司
背景，除非我知道。实际上我是由NVIDIA的研究人员训练的语言模型。不过用户只要求一句话。\n\n可以这样说：我是一个由人工
智能驱动的对话助手，能够帮助你回答问题、提供信息并完成各种文字任务。\n\n或者更简单：我是一个人工智能语言模型，可以
协助你完成写作、翻译、编程、问答等各类任务。\n\n用中文，一句话。'
    },
    {
        'type': 'text',
        'text': '我是一个人工智能语言助手，可以帮你回答问题、提供信息，并协助完成写作、翻译、编程等各类文字任务。'
    }
]